In [5]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    calibration_model,
    SimpleAutoSort
)


In [6]:
# 加载数据（与recordings_30channels_12_month_train.ipynb一致）
files = sorted(os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image"))
recording_list = []
for file in files:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{file}')
    recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
    recording_list.append(recording_recorded.time_slice(start_time=60, end_time=1260))

probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = si.concatenate_recordings(recording_list)
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
# 设置输出文件夹（与recordings_30channels_12_month_train.ipynb中的output_folder一致）
output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/'
combined_output_base = output_folder

# 计算session范围和session名称（与recordings_30channels_12_month_train.ipynb一致）
sampling_frequency = recording_cmr.get_sampling_frequency()

# 计算每个session的采样点范围
segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
segment_num_samples_dict = {}  # {segment_idx: num_samples}
session_names = []  # 存储每个session的名称

# 获取每个session的文件名（去掉扩展名）
for i, file in enumerate(files):
    # 去掉文件扩展名，作为session名称
    session_name = Path(file).stem
    session_names.append(session_name)

# 根据recording_list中每个recording的采样点数计算范围
current_sample = 0
n_segments = len(recording_list)  # session数量等于recording_list的长度

for seg_idx in range(n_segments):
    # 获取该recording的采样点数（在合并前的原始recording）
    segment_num_samples = recording_list[seg_idx].get_num_samples()
    start_sample = current_sample
    end_sample = current_sample + segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)
    segment_num_samples_dict[seg_idx] = segment_num_samples
    
    session_name = session_names[seg_idx] if seg_idx < len(session_names) else f"session_{seg_idx}"
    print(f"Session {seg_idx} ({session_name}): 采样点范围 = [{start_sample}, {end_sample}), 采样点数 = {segment_num_samples}")
    
    current_sample = end_sample

print(f"\n共 {n_segments} 个sessions")

Session 0 (mouse6_012123_natural_image_001): 采样点范围 = [0, 12000000), 采样点数 = 12000000
Session 1 (mouse6_021322_natural_image_001): 采样点范围 = [12000000, 24000000), 采样点数 = 12000000
Session 2 (mouse6_022223_natural_image_001): 采样点范围 = [24000000, 36000000), 采样点数 = 12000000
Session 3 (mouse6_022522_natural_image_001): 采样点范围 = [36000000, 48000000), 采样点数 = 12000000
Session 4 (mouse6_031722_natural_image_001): 采样点范围 = [48000000, 60000000), 采样点数 = 12000000
Session 5 (mouse6_032123_natural_image_001): 采样点范围 = [60000000, 72000000), 采样点数 = 12000000
Session 6 (mouse6_042323_natural_image_001): 采样点范围 = [72000000, 84000000), 采样点数 = 12000000
Session 7 (mouse6_042422_natural_image_001): 采样点范围 = [84000000, 96000000), 采样点数 = 12000000
Session 8 (mouse6_052422_natural_image_001): 采样点范围 = [96000000, 108000000), 采样点数 = 12000000
Session 9 (mouse6_062422_natural_image_001): 采样点范围 = [108000000, 120000000), 采样点数 = 12000000
Session 10 (mouse6_072322_natural_image_001): 采样点范围 = [120000000, 132000000), 采样点数 = 12000000


In [7]:
# 指定训练session和测试sessions
train_session_name = 'mouse6_021322_natural_image_001'
test_session_names = [
    'mouse6_021322_natural_image_001',
    'mouse6_022522_natural_image_001',
    'mouse6_031722_natural_image_001',
    'mouse6_042422_natural_image_001',
    'mouse6_052422_natural_image_001',
    'mouse6_062422_natural_image_001',
    'mouse6_072322_natural_image_001',
    'mouse6_082322_natural_image_001',
    'mouse6_092422_natural_image_001',
    'mouse6_102122_natural_image_001',
    'mouse6_112022_natural_image_001',
    'mouse6_122022_natural_image_001'
]

# 创建单个包含所有30个通道的clique（与训练时一致）
probe = recording_cmr.get_probe()
probe_df = probe.to_dataframe()
all_channel_ids = probe_df['contact_ids'].astype(str).tolist()
cliques = [
    CliqueInfo(
        clique_id=0,
        device_channel_indices=list(range(len(all_channel_ids))),
        contact_ids=all_channel_ids,
        center=(probe_df['x'].mean(), probe_df['y'].mean())
    )
]

print(f"训练session: {train_session_name}")
print(f"测试sessions: {test_session_names}")

训练session: mouse6_021322_natural_image_001
测试sessions: ['mouse6_021322_natural_image_001', 'mouse6_022522_natural_image_001', 'mouse6_031722_natural_image_001', 'mouse6_042422_natural_image_001', 'mouse6_052422_natural_image_001', 'mouse6_062422_natural_image_001', 'mouse6_072322_natural_image_001', 'mouse6_082322_natural_image_001', 'mouse6_092422_natural_image_001', 'mouse6_102122_natural_image_001', 'mouse6_112022_natural_image_001', 'mouse6_122022_natural_image_001']


In [8]:
# Clique级别测试流程（使用训练session的模型测试多个测试sessions）
import torch

# 找到训练session的索引
train_session_idx = None
for idx, name in enumerate(session_names):
    if name == train_session_name:
        train_session_idx = idx
        break

if train_session_idx is None:
    raise ValueError(f"未找到指定的训练session: {train_session_name}")

print(f"训练session: {train_session_name} (index: {train_session_idx})")

# 对每个clique进行处理
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"Processing Clique {clique_id}")
    print(f"{'='*60}")
    
    # 加载训练session的neuron_inf
    train_data_folder = f'{combined_output_base}/clique_{clique_id}/{train_session_name}'
    train_neuron_inf_path = f'{train_data_folder}/neuron_inf.pickle'
    
    if not os.path.exists(train_neuron_inf_path):
        print(f"  警告: 训练session的neuron_inf文件不存在: {train_neuron_inf_path}，跳过clique {clique_id}")
        continue
    
    with open(train_neuron_inf_path, 'rb') as f:
        train_neuron_inf_dict = pickle.load(f)
    
    train_neuron_inf = neuron_inf_dict_to_dataframe(train_neuron_inf_dict)
    print(f"\n训练session {train_session_name}: {len(train_neuron_inf)} 个神经元")
    
    # 对每个测试session进行处理
    for test_session_name in test_session_names:
        # 找到测试session的索引
        test_session_idx = None
        for idx, name in enumerate(session_names):
            if name == test_session_name:
                test_session_idx = idx
                break
        
        if test_session_idx is None:
            print(f"  警告: 未找到指定的测试session: {test_session_name}，跳过")
            continue
        
        print(f"\n{'='*60}")
        print(f"处理测试session: {test_session_name} (index: {test_session_idx})")
        print(f"{'='*60}")
        
        # 加载测试session的neuron_inf
        test_data_folder = f'{combined_output_base}/clique_{clique_id}/{test_session_name}'
        test_neuron_inf_path = f'{test_data_folder}/neuron_inf.pickle'
        
        if not os.path.exists(test_neuron_inf_path):
            print(f"  警告: 测试session的neuron_inf文件不存在: {test_neuron_inf_path}，跳过")
            continue
        
        with open(test_neuron_inf_path, 'rb') as f:
            test_neuron_inf_dict = pickle.load(f)
        
        test_neuron_inf = neuron_inf_dict_to_dataframe(test_neuron_inf_dict)
        print(f"测试session {test_session_name}: {len(test_neuron_inf)} 个神经元")
        
        # 比较训练和测试session的neuron，找出重合、消失、新出现的（直接比较neuron ID）
        print(f"\n{'='*60}")
        print("比较神经元（直接比较neuron ID）")
        print(f"{'='*60}")
        
        train_neuron_ids = set(train_neuron_inf['Neuron'].unique())
        test_neuron_ids = set(test_neuron_inf['Neuron'].unique())
        
        # 统计结果
        matched_neuron_ids = train_neuron_ids & test_neuron_ids  # 重合的神经元
        disappeared_neuron_ids = train_neuron_ids - test_neuron_ids  # 消失的神经元
        new_neuron_ids = test_neuron_ids - train_neuron_ids  # 新出现的神经元
        
        print(f"\n统计结果:")
        print(f"  重合的神经元: {len(matched_neuron_ids)} (在{train_session_name}和{test_session_name}中都存在)")
        if len(matched_neuron_ids) > 0:
            print(f"    重合的神经元列表: {sorted(matched_neuron_ids)}")
        print(f"  消失的神经元: {len(disappeared_neuron_ids)} (在{train_session_name}中存在但在{test_session_name}中不存在)")
        if len(disappeared_neuron_ids) > 0:
            print(f"    消失的神经元列表: {sorted(disappeared_neuron_ids)}")
        print(f"  新出现的神经元: {len(new_neuron_ids)} (在{test_session_name}中存在但在{train_session_name}中不存在)")
        if len(new_neuron_ids) > 0:
            print(f"    新出现的神经元列表: {sorted(new_neuron_ids)}")
        
        # 为test_neuron_inf添加neuron_match列（用于calibration_model的评估）
        test_neuron_inf_matched = test_neuron_inf.copy()
        test_neuron_inf_matched['neuron_match'] = test_neuron_inf_matched['Neuron'].apply(
            lambda x: x if x in matched_neuron_ids else 'unmatch'
        )
        
        # 从recording_cmr中提取该测试session的recording（根据采样点范围）
        start_sample, end_sample = segment_sample_ranges[test_session_idx]
        
        # 从recording_cmr中提取该session的recording
        test_session_recording = recording_cmr.frame_slice(start_frame=start_sample, end_frame=end_sample)
        
        # 获取recording_clique（对于30通道，clique包含所有通道，所以recording_clique就是session_recording）
        test_recording_clique = get_recording_clique(test_session_recording, clique)
        print(f"  测试recording通道数: {len(test_recording_clique.get_channel_ids())}")
        
        test_gt_detect_array_path = f'{test_data_folder}/gt_detect_array.csv'
        test_gt_detect_array = None
        if os.path.exists(test_gt_detect_array_path):
            test_gt_detect_array = pd.read_csv(test_gt_detect_array_path)
        
        # 重复实验5次，每次使用不同的模型权重
        n_repeats = 5
        n_channels = test_recording_clique.get_num_channels()  # 30通道
        samplepoints = 30  # left_sample + right_sample = 10 + 20
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        for repeat_idx in range(1, n_repeats + 1):
            print(f"\n  ===== 重复实验 {repeat_idx}/{n_repeats} (使用 model_{repeat_idx}) =====")
            
            model_save_dir = f'{train_data_folder}/model_{repeat_idx}'
            noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
            label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
            
            # 检查模型文件是否存在
            if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
                print(f"  警告: model_{repeat_idx} 的权重文件不存在，跳过")
                continue
            
            # 加载classification_mapping（与训练时保存的格式一致）
            classification_mapping_path = f'{model_save_dir}/classification_mapping.pkl'
            if not os.path.exists(classification_mapping_path):
                print(f"  警告: classification_mapping.pkl不存在 {classification_mapping_path}，跳过")
                continue
            
            with open(classification_mapping_path, 'rb') as f:
                classification_mapping = pickle.load(f)
            keep_id_list = classification_mapping['label_list']
            
            # 创建模型（使用正确的构造函数参数）
            autosort_model = SimpleAutoSort(
                ch_num=n_channels,
                samplepoints=samplepoints,
                device=device,
                set_shank_id=keep_id_list,
                save_dir=model_save_dir,
                pos_weight_noise=None,  # 这些权重在推理时不需要
                pos_weight_label=None
            )
            
            autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
            autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
            autosort_model.eval()
            
            # 运行calibration
            calibration_results = calibration_model(
                recording_f=test_recording_clique,
                autosort_model=autosort_model,
                train_neuron_inf=train_neuron_inf,
                calibration_duration_seconds=300,
                n_additional_clusters=5,
                detection_params={
                    'thr_min': 3,
                    'thr_max': 30,
                    'distance': 3,
                    'wlen': 5,
                    'prominence': 15,
                    'max_firing_channel': None,
                },
                window_params={
                    'left_sample': 10,
                    'right_sample': 20,
                },
                position_threshold=20.0,
                waveform_similarity_threshold=0.9,
                eval_neuron_inf=test_neuron_inf_matched,
                gt_detect_array=test_gt_detect_array,
                match_mode='per_channel_match',
                device=device
            )
            
            # 保存结果到不同的文件
            output_path = f"{test_data_folder}/calibration_model_{repeat_idx}.pkl"
            with open(output_path, 'wb') as f:
                pickle.dump(calibration_results, f)
            
            print(f"  重复实验 {repeat_idx}/{n_repeats} 完成，结果已保存到: {output_path}")

print("\n所有测试完成！")


训练session: mouse6_021322_natural_image_001 (index: 1)

Processing Clique 0

训练session mouse6_021322_natural_image_001: 38 个神经元

处理测试session: mouse6_021322_natural_image_001 (index: 1)
测试session mouse6_021322_natural_image_001: 38 个神经元

比较神经元（直接比较neuron ID）

统计结果:
  重合的神经元: 38 (在mouse6_021322_natural_image_001和mouse6_021322_natural_image_001中都存在)
    重合的神经元列表: [3, 4, 5, 7, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 38, 41, 46, 47, 48, 49, 50, 51, 52]
  消失的神经元: 0 (在mouse6_021322_natural_image_001中存在但在mouse6_021322_natural_image_001中不存在)
  新出现的神经元: 0 (在mouse6_021322_natural_image_001中存在但在mouse6_021322_natural_image_001中不存在)
  测试recording通道数: 30

  ===== 重复实验 1/5 (使用 model_1) =====
  警告: model_1 的权重文件不存在，跳过

  ===== 重复实验 2/5 (使用 model_2) =====
  警告: model_2 的权重文件不存在，跳过

  ===== 重复实验 3/5 (使用 model_3) =====
  警告: model_3 的权重文件不存在，跳过

  ===== 重复实验 4/5 (使用 model_4) =====
  警告: model_4 的权重文件不存在，跳过

  ===== 重复实验 5/5 (使用 model_5) =====
  警告:

In [11]:
# 计算real_time_processing的耗时（不同time_window，每个重复100次）
import time
import statistics
import importlib
import sys

# 重新加载模块以确保使用最新版本
if 'utils_clique' in sys.modules:
    importlib.reload(sys.modules['utils_clique'])

from utils_clique import real_time_processing, SimpleAutoSort, get_recording_clique, neuron_inf_dict_to_dataframe

# 使用已有的数据路径和设置
# output_folder已经在Cell 1中定义
# recording_cmr, cliques, train_session_name, test_session_names等已经在前面定义

# 设置测试参数
train_session_name = 'mouse6_021322_natural_image_001'
test_session_name = 'mouse6_022522_natural_image_001'  # 测试第一个session
model_repeat = 1  # 使用model_1

# 找到训练和测试session的索引
train_session_idx = None
test_session_idx = None
for idx, name in enumerate(session_names):
    if name == train_session_name:
        train_session_idx = idx
    if name == test_session_name:
        test_session_idx = idx

if train_session_idx is None or test_session_idx is None:
    raise ValueError(f"未找到指定的session")

# 构建路径
clique_id = 0
train_data_folder = f'{combined_output_base}/clique_{clique_id}/{train_session_name}'
test_data_folder = f'{combined_output_base}/clique_{clique_id}/{test_session_name}'
model_save_dir = f'{train_data_folder}/model_{model_repeat}'

# 加载训练neuron_inf
train_neuron_inf_path = f'{train_data_folder}/neuron_inf.pickle'
with open(train_neuron_inf_path, 'rb') as f:
    train_neuron_inf_dict = pickle.load(f)
train_neuron_inf = neuron_inf_dict_to_dataframe(train_neuron_inf_dict)

# 加载calibration_results
calibration_results_path = f'{test_data_folder}/calibration_model_{model_repeat}.pkl'
print(f"加载calibration结果: {calibration_results_path}")
with open(calibration_results_path, 'rb') as f:
    calibration_results = pickle.load(f)

# 从recording_cmr中提取该测试session的recording
start_sample, end_sample = segment_sample_ranges[test_session_idx]
test_session_recording = recording_cmr.frame_slice(start_frame=start_sample, end_frame=end_sample)

# 获取recording_clique
clique = cliques[clique_id]
test_recording_clique = get_recording_clique(test_session_recording, clique)

# 加载模型
print(f"\n加载模型: {model_save_dir}")
n_channels = test_recording_clique.get_num_channels()
samplepoints = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 加载classification_mapping
classification_mapping_path = f'{model_save_dir}/classification_mapping.pkl'
with open(classification_mapping_path, 'rb') as f:
    classification_mapping = pickle.load(f)
keep_id_list = classification_mapping['label_list']

# 创建并加载模型
autosort_model = SimpleAutoSort(
    ch_num=n_channels,
    samplepoints=samplepoints,
    device=device,
    set_shank_id=keep_id_list,
    save_dir=model_save_dir,
    pos_weight_noise=None,
    pos_weight_label=None
)

noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
autosort_model.eval()

print(f"模型加载完成")
print(f"  - Clique ID: {clique_id}")
print(f"  - Channels: {n_channels}")
print(f"  - Device: {device}")

# 设置real_time_processing参数
start_time_seconds = 60.0  # 从60秒开始（calibration之后）
# 测试不同的time_window（毫秒转换为秒）
time_windows_ms = [50, 100, 200, 500, 1000]
time_windows_seconds = [tw / 1000.0 for tw in time_windows_ms]
n_repeats = 100  # 每个time_window重复100次

detection_params = {
    'thr_min': 3,
    'thr_max': 30,
    'distance': 3,
    'wlen': 5,
    'prominence': 15,
    'max_firing_channel': None,
}

window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# 不加载评估数据（只计算时间，不关心准确率）
eval_neuron_inf = None
eval_spike_inf = None

print(f"\n{'=' * 60}")
print(f"开始测试real_time_processing耗时")
print(f"{'=' * 60}")
print(f"测试参数:")
print(f"  - Train session: {train_session_name}")
print(f"  - Test session: {test_session_name}")
print(f"  - Start time: {start_time_seconds} seconds")
print(f"  - Time windows: {time_windows_ms} ms ({time_windows_seconds} seconds)")
print(f"  - Repeats per window: {n_repeats}")
print(f"  - Total tests: {len(time_windows_ms) * n_repeats}")
print(f"{'=' * 60}\n")

# 存储所有结果
all_results = {}

# 对每个time_window进行测试
for tw_idx, time_window_seconds in enumerate(time_windows_seconds):
    time_window_ms = time_windows_ms[tw_idx]
    print(f"\n{'=' * 60}")
    print(f"测试 Time Window: {time_window_ms} ms ({time_window_seconds} seconds)")
    print(f"{'=' * 60}")
    
    # 计算需要处理的总时长（至少处理1秒的数据，确保有足够的窗口）
    total_duration_seconds = max(1.0, time_window_seconds * 10)  # 至少10个窗口
    
    elapsed_times = []
    
    # 重复100次
    for repeat_idx in range(n_repeats):
        if (repeat_idx + 1) % 10 == 0:
            print(f"  进度: {repeat_idx + 1}/{n_repeats}")
        
        # 测量单次耗时
        start_time = time.time()
        
        processing_results = real_time_processing(
            recording_f=test_recording_clique,
            autosort_model=autosort_model,
            calibration_results=calibration_results,
            start_time_seconds=start_time_seconds,
            time_window_seconds=time_window_seconds,
            total_duration_seconds=total_duration_seconds,
            detection_params=detection_params,
            window_params=window_params,
            eval_neuron_inf=eval_neuron_inf,
            eval_spike_inf=eval_spike_inf,
            device=device,
            batch_size=512,  # 优化：增大批处理大小
            verbose=False,  # 优化：禁用详细输出
            save_noise_features=False,  # 优化：不保存noise visualization数据
        )
        
        end_time = time.time()
        elapsed_time = end_time - start_time
        elapsed_times.append(elapsed_time)
    
    # 计算统计信息
    mean_time = statistics.mean(elapsed_times)
    median_time = statistics.median(elapsed_times)
    min_time = min(elapsed_times)
    max_time = max(elapsed_times)
    std_time = statistics.stdev(elapsed_times) if len(elapsed_times) > 1 else 0.0
    
    # 计算实时因子
    n_windows = int(total_duration_seconds / time_window_seconds)
    realtime_factor = total_duration_seconds / mean_time
    
    # 存储结果
    all_results[time_window_ms] = {
        'time_window_ms': time_window_ms,
        'time_window_seconds': time_window_seconds,
        'total_duration_seconds': total_duration_seconds,
        'n_windows': n_windows,
        'n_repeats': n_repeats,
        'elapsed_times': elapsed_times,
        'mean_time': mean_time,
        'median_time': median_time,
        'min_time': min_time,
        'max_time': max_time,
        'std_time': std_time,
        'realtime_factor': realtime_factor,
        'mean_time_per_window': mean_time / n_windows,
    }
    
    # 输出结果
    print(f"\n  结果统计 ({time_window_ms} ms):")
    print(f"    - 平均耗时: {mean_time:.4f} 秒")
    print(f"    - 中位数耗时: {median_time:.4f} 秒")
    print(f"    - 最小耗时: {min_time:.4f} 秒")
    print(f"    - 最大耗时: {max_time:.4f} 秒")
    print(f"    - 标准差: {std_time:.4f} 秒")
    print(f"    - 处理时长: {total_duration_seconds:.4f} 秒 ({n_windows} 个窗口)")
    print(f"    - 实时因子: {realtime_factor:.2f}x")
    print(f"    - 每个窗口平均耗时: {mean_time / n_windows:.4f} 秒")

# 输出汇总结果
print(f"\n{'=' * 60}")
print(f"汇总结果")
print(f"{'=' * 60}")
print(f"{'Time Window (ms)':<20} {'Mean Time (s)':<15} {'Std (s)':<12} {'Realtime Factor':<15} {'Time/Window (s)':<15}")
print(f"{'-' * 80}")
for tw_ms in time_windows_ms:
    result = all_results[tw_ms]
    print(f"{tw_ms:<20} {result['mean_time']:<15.4f} {result['std_time']:<12.4f} {result['realtime_factor']:<15.2f} {result['mean_time_per_window']:<15.4f}")

# 保存结果
output_path = f"{test_data_folder}/realtime_processing_timing_results.pkl"
with open(output_path, 'wb') as f:
    pickle.dump({
        'all_results': all_results,
        'parameters': {
            'start_time_seconds': start_time_seconds,
            'time_windows_ms': time_windows_ms,
            'time_windows_seconds': time_windows_seconds,
            'n_repeats': n_repeats,
            'detection_params': detection_params,
            'window_params': window_params,
            'train_session_name': train_session_name,
            'test_session_name': test_session_name,
            'clique_id': clique_id,
            'device': str(device),
        }
    }, f)
print(f"\n✓ 所有结果已保存到: {output_path}")


加载calibration结果: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim/clique_0/mouse6_022522_natural_image_001/calibration_model_1.pkl

加载模型: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim/clique_0/mouse6_021322_natural_image_001/model_1
模型加载完成
  - Clique ID: 0
  - Channels: 30
  - Device: cuda

开始测试real_time_processing耗时
测试参数:
  - Train session: mouse6_021322_natural_image_001
  - Test session: mouse6_022522_natural_image_001
  - Start time: 60.0 seconds
  - Time windows: [50, 100, 200, 500, 1000] ms ([0.05, 0.1, 0.2, 0.5, 1.0] seconds)
  - Repeats per window: 100
  - Total tests: 500


测试 Time Window: 50 ms (0.05 seconds)

Processing window 1 (60.0s - 60.0s)

Processing window 2 (60.0s - 60.1s)

Processing window 3 (60.1s - 60.1s)

Processing window 4 (60.1s - 60.2s)

Processing window 5 (60.2s - 60.2s)

Processing window 6 (60.2s - 60.3s)

Processing window 7 (60.3s - 60.4s)

Processing window 8 (60.4s - 60.4s)

Processin